# Forensic Twins — Kaggle 2×T4

Notebook tự chứa toàn bộ source code của baseline `genimage_lite_50k_provisional_v1`.

Kaggle Inputs duy nhất:

- `imagenet-object-localization-challenge`;
- `coco-2017-dataset`;
- `tiny-genimage` (chỉ cần khi evaluation).

`fre.pt` được tự động tải từ official FSD release. Hãy bật **Internet** cho notebook.

> Để không mất tiến độ khi session bị xóa, hãy tạo trước một private Kaggle Dataset cho checkpoint rồi điền `CHECKPOINT_DATASET_SLUG`. Checkpoint local được lưu mỗi 250 step và cuối mỗi epoch; checkpoint cuối epoch được version lên Dataset này.


In [ ]:
# ========================= USER CONFIG =========================
# Private Dataset phải được tạo trước, dạng: username/dataset-name
CHECKPOINT_DATASET_SLUG = ""  # Ví dụ: "myname/forensic-twins-checkpoints"

# Chỉ bật sau khi Tiny-GenImage đã có trong Kaggle Input.
RUN_EVALUATION = False
TINY_GENIMAGE_ROOT = ""  # Để trống để tự tìm
MAX_EVAL_PER_CLASS = None  # Có thể đặt 500 để chạy thử nhanh

INPUT_ROOT = "/kaggle/input"
OUTPUT_ROOT = "/kaggle/working/forensic_twins_original"
WORKING_CODE_DIR = "/kaggle/working/forensic_twins_code"
print("Configuration loaded.")


## 1. Kiểm tra GPU và môi trường

Notebook dừng ngay nếu Kaggle không cung cấp đúng hai CUDA device. T4 dùng FP16 vì không có native BF16.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import PIL
import numpy as np
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Pillow:", PIL.__version__)
print("NumPy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"  GPU {index}: {torch.cuda.get_device_name(index)}")

assert torch.cuda.is_available(), "Hãy bật GPU Accelerator trong Kaggle."
assert torch.cuda.device_count() == 2, (
    f"Config yêu cầu đúng 2 GPU, hiện có {torch.cuda.device_count()}. "
    "Chọn accelerator GPU T4 x2."
)
subprocess.run(["nvidia-smi"], check=True)

## 2. Bung source code được nhúng trong notebook

Notebook mang sẵn training code, model, config và tests. Cell kế tiếp giải nén chúng vào `/kaggle/working`; không cần gắn thêm code Dataset và không cần clone GitHub.


In [ ]:
import base64
import io
import json
import zipfile

SOURCE_ARCHIVE_B64 = (
    "..."
    "SwUGAAAAAAsACwDTAgAAnKoAAAAA"
)
input_root = Path(INPUT_ROOT)
working_code = Path(WORKING_CODE_DIR)
if working_code.exists():
    shutil.rmtree(working_code)
working_code.mkdir(parents=True, exist_ok=True)
archive_bytes = base64.b64decode(SOURCE_ARCHIVE_B64)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    archive.extractall(working_code)
CODE_DIR = working_code
CONFIG_PATH = CODE_DIR / "configs" / "kaggle_2xt4.json"
if CHECKPOINT_DATASET_SLUG:
    runtime_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    runtime_config["checkpoint"]["kaggle_dataset_slug"] = CHECKPOINT_DATASET_SLUG
    CONFIG_PATH.write_text(json.dumps(runtime_config, indent=2) + "\n", encoding="utf-8")
    print("Checkpoint slug written into runtime config:", CHECKPOINT_DATASET_SLUG)
required = [
    CODE_DIR / "kaggle_launcher.py",
    CODE_DIR / "train_forensic_twins_kaggle.py",
    CODE_DIR / "src" / "core.py",
    CONFIG_PATH,
]
assert all(path.is_file() for path in required), required
print("Embedded source extracted to:", CODE_DIR)
print("Embedded archive bytes:", len(archive_bytes))


## 3. Preflight Kaggle Inputs

Cell này chỉ kiểm tra cấu trúc thư mục, không đếm toàn bộ ảnh. COCO hiện tại không có `unlabeled2017`, vì vậy code dùng fallback 25.000 ảnh từ `train2017` và ghi rõ deviation trong manifest summary.

In [ ]:
# Exact mount paths derived from the three Kaggle source URLs.
input_root = Path(INPUT_ROOT)

imagenet_train = (
    input_root / "competitions" / "imagenet-object-localization-challenge"
    / "ILSVRC" / "Data" / "CLS-LOC" / "train"
)
imagenet_index = (
    input_root / "competitions" / "imagenet-object-localization-challenge"
    / "LOC_train_solution.csv"
)
coco_root = (
    input_root / "datasets" / "awsaf49" / "coco-2017-dataset" / "coco2017"
)
coco_train = coco_root / "train2017"
coco_unlabeled = coco_root / "unlabeled2017"
coco_index_candidates = [
    coco_root / "annotations" / "captions_train2017.json",
    coco_root / "annotations" / "instances_train2017.json",
]
tiny_genimage_root = input_root / "datasets" / "yangsangtai" / "tiny-genimage"

assert imagenet_train.is_dir(), f"Thiếu ImageNet train: {imagenet_train}"
assert coco_train.is_dir(), f"Thiếu COCO train2017: {coco_train}"
assert tiny_genimage_root.is_dir(), f"Thiếu Tiny-GenImage: {tiny_genimage_root}"

assert imagenet_index.is_file(), f"Missing ImageNet index: {imagenet_index}"
assert any(path.is_file() for path in coco_index_candidates), (
    f"Missing COCO train index under: {coco_root / 'annotations'}"
)

# Only list the 1,000 class folders; never recursively walk image files.
with os.scandir(imagenet_train) as entries:
    class_count = sum(1 for entry in entries if entry.is_dir())
assert class_count == 1000, f"Cần 1000 ImageNet class folders, tìm thấy {class_count}"

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
print("ImageNet train:", imagenet_train)
print("ImageNet index:", imagenet_index)
print("ImageNet classes:", class_count)
print("COCO train2017:", coco_train)
print("COCO index:", next(path for path in coco_index_candidates if path.is_file()))
print("COCO unlabeled2017 exists:", coco_unlabeled.is_dir())
print("Tiny-GenImage:", tiny_genimage_root)
print("FRE: auto-download from official FSD release (Internet must be enabled)")
print("Output root:", OUTPUT_ROOT)


In [ ]:
# Xác minh quyền truy cập private checkpoint Dataset trước khi train.
# Nếu cần, tạo Kaggle Secrets có tên KAGGLE_USERNAME và KAGGLE_KEY.
def kaggle_dataset_status(slug: str):
    return subprocess.run(
        ["kaggle", "datasets", "status", slug],
        capture_output=True, text=True, timeout=120,
    )

if CHECKPOINT_DATASET_SLUG:
    status = kaggle_dataset_status(CHECKPOINT_DATASET_SLUG)
    if status.returncode != 0:
        try:
            from kaggle_secrets import UserSecretsClient
            secrets = UserSecretsClient()
            os.environ.setdefault("KAGGLE_USERNAME", secrets.get_secret("KAGGLE_USERNAME"))
            os.environ.setdefault("KAGGLE_KEY", secrets.get_secret("KAGGLE_KEY"))
        except Exception as error:
            print("Không load được Kaggle API secrets:", error)
        status = kaggle_dataset_status(CHECKPOINT_DATASET_SLUG)
    if status.returncode != 0:
        raise RuntimeError(
            "Không truy cập được checkpoint Dataset. Hãy tạo private Dataset trước, "
            "kiểm tra slug và thêm KAGGLE_USERNAME/KAGGLE_KEY vào Kaggle Secrets.\n"
            + (status.stderr or status.stdout)
        )
    print(status.stdout.strip() or f"Checkpoint Dataset OK: {CHECKPOINT_DATASET_SLUG}")

    # Tự tải checkpoint mới nhất bằng Kaggle API. Không cần Add Input checkpoint Dataset.
    checkpoint_dir = Path(OUTPUT_ROOT) / "artifacts" / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    if not any(checkpoint_dir.glob("*.ckpt")):
        download = subprocess.run(
            [
                "kaggle", "datasets", "download", "-d", CHECKPOINT_DATASET_SLUG,
                "-p", str(checkpoint_dir), "--unzip", "--force",
            ],
            capture_output=True, text=True, timeout=1800,
        )
        if download.returncode == 0:
            print(download.stdout.strip())
            print("Downloaded checkpoints:", [p.name for p in checkpoint_dir.glob("*.ckpt")])
        else:
            print("Checkpoint Dataset chưa có checkpoint để tải (run đầu tiên là bình thường).")
            print(download.stderr or download.stdout)
else:
    print("WARNING: chưa cấu hình persistent checkpoint Dataset.")


## 4. Smoke tests

Kiểm tra model, loss, crop, FRE loading, checkpoint và codec-matching trước khi tiêu tốn GPU-hour.

In [ ]:
subprocess.run([sys.executable, str(CODE_DIR / "tests" / "test_smoke.py")], cwd=CODE_DIR, check=True)
subprocess.run([sys.executable, str(CODE_DIR / "tests" / "test_evaluation.py")], cwd=CODE_DIR, check=True)
subprocess.run([sys.executable, str(CODE_DIR / "tests" / "test_manifest.py")], cwd=CODE_DIR, check=True)


## 5. Build manifest real-only 50K

Kết quả phải có đúng 50.000 dòng: 25 ảnh × 1.000 ImageNet class và 25.000 ảnh COCO. Ảnh được audit bằng magic format, kích thước, SHA-256 và JPEG profile.

In [ ]:
manifest_command = [
    sys.executable, str(CODE_DIR / "kaggle_launcher.py"), "manifest",
    "--config", str(CONFIG_PATH),
    "--input-root", INPUT_ROOT,
    "--output-root", OUTPUT_ROOT,
]
if CHECKPOINT_DATASET_SLUG:
    manifest_command.extend(["--kaggle-dataset-slug", CHECKPOINT_DATASET_SLUG])
print("Running:", " ".join(manifest_command))
subprocess.run(manifest_command, cwd=CODE_DIR, check=True)


In [ ]:
import json

manifest_path = Path(OUTPUT_ROOT) / "manifests" / "train_real_50k.csv"
summary_path = manifest_path.with_suffix(".summary.json")
assert manifest_path.is_file() and summary_path.is_file()
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["rows"] == 50_000, summary
assert summary["imagenet_rows"] == 25_000, summary
assert summary["coco_rows"] == 25_000, summary
print(json.dumps(summary, indent=2, ensure_ascii=False))

## 6. Train hoặc auto-resume trên 2×T4

Cấu hình: FP16, micro-batch 64/GPU, global batch 128, LR 0.0006, 80 epoch. Lệnh luôn dùng `resume=auto`: ưu tiên checkpoint mới nhất hợp lệ trong `/kaggle/working`, sau đó tìm checkpoint Dataset được attach dưới `/kaggle/input`.

Nếu `CHECKPOINT_DATASET_SLUG` trống, training vẫn chạy nhưng chỉ có checkpoint local; notebook sẽ in cảnh báo rõ ràng.

In [ ]:
if not CHECKPOINT_DATASET_SLUG:
    print("WARNING: CHECKPOINT_DATASET_SLUG đang trống. Checkpoint sẽ không sống sót nếu Kaggle xóa session.")

train_command = [
    sys.executable, str(CODE_DIR / "kaggle_launcher.py"), "train",
    "--config", str(CONFIG_PATH),
    "--input-root", INPUT_ROOT,
    "--output-root", OUTPUT_ROOT,
    "--resume", "auto",
]
if CHECKPOINT_DATASET_SLUG:
    train_command.extend(["--kaggle-dataset-slug", CHECKPOINT_DATASET_SLUG])
print("Running:", " ".join(train_command))
subprocess.run(train_command, cwd=CODE_DIR, check=True)

## 7. Kiểm tra checkpoint và log

Có thể chạy cell này sau khi training kết thúc hoặc sau khi dừng notebook. `last.ckpt` phải có sidecar SHA-256.

In [ ]:
checkpoint_dir = Path(OUTPUT_ROOT) / "artifacts" / "checkpoints"
last_checkpoint = checkpoint_dir / "last.ckpt"
last_checksum = checkpoint_dir / "last.ckpt.sha256"
print("Checkpoint directory:", checkpoint_dir)
for path in sorted(checkpoint_dir.glob("*")) if checkpoint_dir.exists() else []:
    print(f"  {path.name:32s} {path.stat().st_size / (1024**2):9.2f} MiB")
assert last_checkpoint.is_file(), "Chưa có last.ckpt"
assert last_checksum.is_file(), "Chưa có checksum sidecar"
print("Checksum:", last_checksum.read_text(encoding="utf-8").strip())

log_path = Path(OUTPUT_ROOT) / "artifacts" / "logs" / "train_metrics.csv"
if log_path.is_file():
    print("Last log rows:")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-6:]))

## 8. Fit GMM và đánh giá Tiny-GenImage (tùy chọn)

Chỉ chạy khi `RUN_EVALUATION=True` và đã add Tiny-GenImage. Pipeline trích CLS 192-D cho 50K real, fit Gaussian `K=1` full covariance, rồi báo cả native-confounded và fake-codec-matched AUROC.

In [ ]:
if RUN_EVALUATION:
    eval_command = [
        sys.executable, str(CODE_DIR / "kaggle_launcher.py"), "gmm",
        "--config", str(CONFIG_PATH),
        "--input-root", INPUT_ROOT,
        "--output-root", OUTPUT_ROOT,
    ]
    if TINY_GENIMAGE_ROOT:
        eval_command.extend(["--tiny-root", TINY_GENIMAGE_ROOT])
    if MAX_EVAL_PER_CLASS is not None:
        eval_command.extend(["--max-eval-per-class", str(MAX_EVAL_PER_CLASS)])
    print("Running:", " ".join(eval_command))
    subprocess.run(eval_command, cwd=CODE_DIR, check=True)
else:
    print("Evaluation skipped. Set RUN_EVALUATION=True after attaching Tiny-GenImage.")

In [ ]:
metrics_dir = Path(OUTPUT_ROOT) / "artifacts" / "metrics"
if metrics_dir.is_dir():
    for path in sorted(metrics_dir.glob("*_summary.json")):
        print(f"\n===== {path.name} =====")
        print(path.read_text(encoding="utf-8"))
comparison = metrics_dir / "codec_comparison.json"
if comparison.is_file():
    print("\n===== codec_comparison.json =====")
    print(comparison.read_text(encoding="utf-8"))

## Khôi phục ở session tiếp theo

1. Add lại đúng ba image Inputs: ImageNet, COCO và Tiny-GenImage.
2. Giữ nguyên `CHECKPOINT_DATASET_SLUG`, config và dữ liệu.
3. Notebook tự tải checkpoint mới nhất bằng Kaggle API; không cần Add Input checkpoint Dataset.
4. Chạy lại đến cell training với `--resume auto`.
5. Code xác minh checksum, config hash, manifest hash và FRE hash trước khi restore model, projector, optimizer, scheduler, AMP scaler và RNG state.
